# Common Base vs LoRA 模型对比分析

对比 Qwen2.5-VL-3B 在 **Common 图片**（Ground Truth: In-context）上的表现：
- **Base Model**: 原始预训练模型
- **LoRA Model**: 经过 LoRA 微调的模型

In [ ]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 11

# 数据目录
DATA_DIR = config.FOCUS_DATA_DIR

In [ ]:
import sys, os
sys.path.append(os.path.join(os.path.abspath(''), '..', '..'))
import config

## 1. 加载数据

In [ ]:
import os

# 加载推理结果
base_results = pd.read_csv(os.path.join(DATA_DIR, 'common_all_inference_results_base.csv'))
lora_results = pd.read_csv(os.path.join(DATA_DIR, 'common_all_inference_results_lora.csv'))

# 加载评估指标
with open(os.path.join(DATA_DIR, 'common_all_inference_metrics_base.json'), 'r') as f:
    base_metrics = json.load(f)
with open(os.path.join(DATA_DIR, 'common_all_inference_metrics_lora.json'), 'r') as f:
    lora_metrics = json.load(f)

# 加载 pair 统计
base_pair_stats = pd.read_csv(os.path.join(DATA_DIR, 'common_pair_statistics_base.csv'))
lora_pair_stats = pd.read_csv(os.path.join(DATA_DIR, 'common_pair_statistics_lora.csv'))

print(f"Base Results: {len(base_results)} samples")
print(f"LoRA Results: {len(lora_results)} samples")

## 2. 整体性能对比

In [ ]:
# 创建对比表格
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Total Samples', 'Unknown Predictions'],
    'Base Model': [
        f"{base_metrics['accuracy']:.2%}",
        f"{base_metrics['precision']:.4f}",
        f"{base_metrics['recall']:.4f}",
        f"{base_metrics['f1']:.4f}",
        base_metrics['total'],
        base_metrics['unknown']
    ],
    'LoRA Model': [
        f"{lora_metrics['accuracy']:.2%}",
        f"{lora_metrics['precision']:.4f}",
        f"{lora_metrics['recall']:.4f}",
        f"{lora_metrics['f1']:.4f}",
        lora_metrics['total'],
        lora_metrics['unknown']
    ]
})

# 计算差异
diff_acc = lora_metrics['accuracy'] - base_metrics['accuracy']
comparison['Difference'] = [
    f"{diff_acc:+.2%}",
    f"{lora_metrics['precision'] - base_metrics['precision']:+.4f}",
    f"{lora_metrics['recall'] - base_metrics['recall']:+.4f}",
    f"{lora_metrics['f1'] - base_metrics['f1']:+.4f}",
    '-',
    lora_metrics['unknown'] - base_metrics['unknown']
]

print("=" * 70)
print("📊 整体性能对比 (Common Images - Ground Truth: In-context)")
print("=" * 70)
display(comparison)

In [ ]:
# 可视化整体指标
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 主要指标对比
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1']
base_values = [base_metrics['accuracy'], base_metrics['precision'], base_metrics['recall'], base_metrics['f1']]
lora_values = [lora_metrics['accuracy'], lora_metrics['precision'], lora_metrics['recall'], lora_metrics['f1']]

x = np.arange(len(metrics_names))
width = 0.35

bars1 = axes[0].bar(x - width/2, base_values, width, label='Base', color='#3498db', alpha=0.8)
bars2 = axes[0].bar(x + width/2, lora_values, width, label='LoRA', color='#e74c3c', alpha=0.8)

axes[0].set_ylabel('Score')
axes[0].set_title('Common Images: Base vs LoRA Performance')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_names)
axes[0].legend()
axes[0].set_ylim(0, 1.1)

# 添加数值标签
for bar in bars1:
    axes[0].annotate(f'{bar.get_height():.2%}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=9)
for bar in bars2:
    axes[0].annotate(f'{bar.get_height():.2%}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=9)

# 右图: 预测分布
base_pred_counts = base_results['prediction'].value_counts()
lora_pred_counts = lora_results['prediction'].value_counts()

labels = ['In-context', 'Out-of-context', 'Unknown']
base_preds = [base_pred_counts.get(l, 0) for l in labels]
lora_preds = [lora_pred_counts.get(l, 0) for l in labels]

x = np.arange(len(labels))
bars1 = axes[1].bar(x - width/2, base_preds, width, label='Base', color='#3498db', alpha=0.8)
bars2 = axes[1].bar(x + width/2, lora_preds, width, label='LoRA', color='#e74c3c', alpha=0.8)

axes[1].set_ylabel('Count')
axes[1].set_title('Prediction Distribution (Ground Truth: In-context)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].legend()

for bar in bars1:
    axes[1].annotate(f'{int(bar.get_height())}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=9)
for bar in bars2:
    axes[1].annotate(f'{int(bar.get_height())}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 3. 混淆矩阵对比

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Base 混淆矩阵
base_cm = np.array(base_metrics['confusion_matrix'])
sns.heatmap(base_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: In-ctx', 'Pred: Out-ctx'],
            yticklabels=['True: In-ctx', 'True: Out-ctx'])
axes[0].set_title(f'Base Model (Acc: {base_metrics["accuracy"]:.2%})')

# LoRA 混淆矩阵
lora_cm = np.array(lora_metrics['confusion_matrix'])
sns.heatmap(lora_cm, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['Pred: In-ctx', 'Pred: Out-ctx'],
            yticklabels=['True: In-ctx', 'True: Out-ctx'])
axes[1].set_title(f'LoRA Model (Acc: {lora_metrics["accuracy"]:.2%})')

plt.tight_layout()
plt.show()

# 解读
print("\n📋 混淆矩阵解读 (对于 Common 图片，Ground Truth 全部是 In-context):")
print(f"\nBase Model:")
print(f"  - 正确预测 In-context: {base_cm[0,0]} ({base_cm[0,0]/base_cm.sum()*100:.1f}%)")
print(f"  - 错误预测为 Out-of-context (False Negative): {base_cm[0,1]} ({base_cm[0,1]/base_cm.sum()*100:.1f}%)")

print(f"\nLoRA Model:")
print(f"  - 正确预测 In-context: {lora_cm[0,0]} ({lora_cm[0,0]/lora_cm.sum()*100:.1f}%)")
print(f"  - 错误预测为 Out-of-context (False Negative): {lora_cm[0,1]} ({lora_cm[0,1]/lora_cm.sum()*100:.1f}%)")

## 4. Category-Location Pair 分析

In [ ]:
# 合并 base 和 lora 的 pair 统计
pair_comparison = base_pair_stats[['category', 'location', 'total', 'accuracy']].copy()
pair_comparison.columns = ['category', 'location', 'total', 'base_acc']
pair_comparison['lora_acc'] = lora_pair_stats['accuracy'].values
pair_comparison['diff'] = pair_comparison['lora_acc'] - pair_comparison['base_acc']

# 按差异排序
pair_comparison_sorted = pair_comparison.sort_values('diff', ascending=False)

print("=" * 80)
print("📊 Category-Location Pair 准确率对比 (按改进幅度排序)")
print("=" * 80)
print(f"{'Category':<12} {'Location':<12} {'Total':>8} {'Base Acc':>12} {'LoRA Acc':>12} {'Diff':>10}")
print("-" * 80)
for _, row in pair_comparison_sorted.iterrows():
    diff_str = f"{row['diff']:+.2%}"
    if row['diff'] > 0:
        diff_str = f"🟢 {diff_str}"
    elif row['diff'] < 0:
        diff_str = f"🔴 {diff_str}"
    else:
        diff_str = f"⚪ {diff_str}"
    print(f"{row['category']:<12} {row['location']:<12} {row['total']:>8} {row['base_acc']:>12.2%} {row['lora_acc']:>12.2%} {diff_str:>15}")

In [ ]:
# 可视化 pair 对比
fig, ax = plt.subplots(figsize=(14, 8))

pair_comparison_sorted['pair'] = pair_comparison_sorted['category'] + '-' + pair_comparison_sorted['location']
pairs = pair_comparison_sorted['pair'].values
x = np.arange(len(pairs))
width = 0.35

bars1 = ax.bar(x - width/2, pair_comparison_sorted['base_acc'], width, label='Base', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, pair_comparison_sorted['lora_acc'], width, label='LoRA', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Accuracy')
ax.set_title('Category-Location Pair Accuracy: Base vs LoRA')
ax.set_xticks(x)
ax.set_xticklabels(pairs, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.15)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50% baseline')

plt.tight_layout()
plt.show()

In [ ]:
# 改进/退步分析
improved = pair_comparison[pair_comparison['diff'] > 0]
degraded = pair_comparison[pair_comparison['diff'] < 0]
unchanged = pair_comparison[pair_comparison['diff'] == 0]

print("\n📈 改进分析:")
print(f"  - 改进的 pairs: {len(improved)} ({len(improved)/len(pair_comparison)*100:.1f}%)")
print(f"  - 退步的 pairs: {len(degraded)} ({len(degraded)/len(pair_comparison)*100:.1f}%)")
print(f"  - 无变化的 pairs: {len(unchanged)} ({len(unchanged)/len(pair_comparison)*100:.1f}%)")

if len(improved) > 0:
    print(f"\n🏆 改进最大的 pairs:")
    for _, row in improved.nlargest(3, 'diff').iterrows():
        print(f"   {row['category']}-{row['location']}: {row['base_acc']:.2%} → {row['lora_acc']:.2%} ({row['diff']:+.2%})")

if len(degraded) > 0:
    print(f"\n⚠️ 退步最大的 pairs:")
    for _, row in degraded.nsmallest(3, 'diff').iterrows():
        print(f"   {row['category']}-{row['location']}: {row['base_acc']:.2%} → {row['lora_acc']:.2%} ({row['diff']:+.2%})")

## 5. 样本级对比分析

In [ ]:
# 合并结果进行样本级比较
merged = base_results[['filename', 'category', 'location', 'prediction', 'correct']].copy()
merged.columns = ['filename', 'category', 'location', 'base_pred', 'base_correct']
merged['lora_pred'] = lora_results['prediction'].values
merged['lora_correct'] = lora_results['correct'].values

# 分类情况
both_correct = merged[(merged['base_correct']) & (merged['lora_correct'])]
both_wrong = merged[(~merged['base_correct']) & (~merged['lora_correct'])]
base_only_correct = merged[(merged['base_correct']) & (~merged['lora_correct'])]
lora_only_correct = merged[(~merged['base_correct']) & (merged['lora_correct'])]

print("=" * 60)
print("📊 样本级对比分析")
print("=" * 60)
print(f"\n总样本数: {len(merged)}")
print(f"\n两模型都正确 ✓✓: {len(both_correct)} ({len(both_correct)/len(merged)*100:.1f}%)")
print(f"两模型都错误 ✗✗: {len(both_wrong)} ({len(both_wrong)/len(merged)*100:.1f}%)")
print(f"仅 Base 正确 ✓✗: {len(base_only_correct)} ({len(base_only_correct)/len(merged)*100:.1f}%)")
print(f"仅 LoRA 正确 ✗✓: {len(lora_only_correct)} ({len(lora_only_correct)/len(merged)*100:.1f}%)")

In [ ]:
# 饼图展示
fig, ax = plt.subplots(figsize=(8, 8))

sizes = [len(both_correct), len(both_wrong), len(base_only_correct), len(lora_only_correct)]
labels = [f'Both Correct\n({sizes[0]})', f'Both Wrong\n({sizes[1]})', 
          f'Base Only\n({sizes[2]})', f'LoRA Only\n({sizes[3]})']
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']
explode = (0, 0.05, 0, 0)

ax.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
       shadow=True, startangle=90)
ax.set_title('Sample-Level Prediction Comparison')

plt.tight_layout()
plt.show()

In [ ]:
# 两模型都错误的样本 - 按 category-location 分组
if len(both_wrong) > 0:
    print("\n🔴 两模型都错误的样本分布:")
    wrong_dist = both_wrong.groupby(['category', 'location']).size().reset_index(name='count')
    wrong_dist = wrong_dist.sort_values('count', ascending=False)
    for _, row in wrong_dist.iterrows():
        print(f"   {row['category']}-{row['location']}: {row['count']}")

In [ ]:
# 仅 LoRA 正确的样本 (LoRA 改进的地方)
if len(lora_only_correct) > 0:
    print("\n🟢 仅 LoRA 正确的样本 (LoRA 改进的地方):")
    lora_improve = lora_only_correct.groupby(['category', 'location']).size().reset_index(name='count')
    lora_improve = lora_improve.sort_values('count', ascending=False)
    for _, row in lora_improve.iterrows():
        print(f"   {row['category']}-{row['location']}: {row['count']}")

In [ ]:
# 仅 Base 正确的样本 (LoRA 退步的地方)
if len(base_only_correct) > 0:
    print("\n🔵 仅 Base 正确的样本 (LoRA 退步的地方):")
    base_better = base_only_correct.groupby(['category', 'location']).size().reset_index(name='count')
    base_better = base_better.sort_values('count', ascending=False)
    for _, row in base_better.iterrows():
        print(f"   {row['category']}-{row['location']}: {row['count']}")

## 6. 按 Category 分析

In [ ]:
# 按 category 聚合
base_by_cat = base_results.groupby('category')['correct'].agg(['sum', 'count']).reset_index()
base_by_cat.columns = ['category', 'base_correct', 'total']
base_by_cat['base_acc'] = base_by_cat['base_correct'] / base_by_cat['total']

lora_by_cat = lora_results.groupby('category')['correct'].agg(['sum', 'count']).reset_index()
lora_by_cat.columns = ['category', 'lora_correct', 'total']
lora_by_cat['lora_acc'] = lora_by_cat['lora_correct'] / lora_by_cat['total']

cat_comparison = base_by_cat.merge(lora_by_cat[['category', 'lora_acc']], on='category')
cat_comparison['diff'] = cat_comparison['lora_acc'] - cat_comparison['base_acc']
cat_comparison = cat_comparison.sort_values('diff', ascending=False)

print("=" * 60)
print("📊 按 Category 的准确率对比")
print("=" * 60)
print(f"{'Category':<15} {'Total':>8} {'Base Acc':>12} {'LoRA Acc':>12} {'Diff':>10}")
print("-" * 60)
for _, row in cat_comparison.iterrows():
    diff_str = f"{row['diff']:+.2%}"
    if row['diff'] > 0:
        diff_str = f"🟢 {diff_str}"
    elif row['diff'] < 0:
        diff_str = f"🔴 {diff_str}"
    print(f"{row['category']:<15} {row['total']:>8} {row['base_acc']:>12.2%} {row['lora_acc']:>12.2%} {diff_str:>15}")

In [ ]:
# Category 准确率对比图
fig, ax = plt.subplots(figsize=(12, 6))

categories = cat_comparison['category'].values
x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width/2, cat_comparison['base_acc'], width, label='Base', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, cat_comparison['lora_acc'], width, label='LoRA', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Accuracy')
ax.set_title('Per-Category Accuracy: Base vs LoRA')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.15)

# 添加数值
for bar in bars1:
    ax.annotate(f'{bar.get_height():.0%}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.0%}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 7. 按 Location 分析

In [ ]:
# 按 location 聚合
base_by_loc = base_results.groupby('location')['correct'].agg(['sum', 'count']).reset_index()
base_by_loc.columns = ['location', 'base_correct', 'total']
base_by_loc['base_acc'] = base_by_loc['base_correct'] / base_by_loc['total']

lora_by_loc = lora_results.groupby('location')['correct'].agg(['sum', 'count']).reset_index()
lora_by_loc.columns = ['location', 'lora_correct', 'total']
lora_by_loc['lora_acc'] = lora_by_loc['lora_correct'] / lora_by_loc['total']

loc_comparison = base_by_loc.merge(lora_by_loc[['location', 'lora_acc']], on='location')
loc_comparison['diff'] = loc_comparison['lora_acc'] - loc_comparison['base_acc']
loc_comparison = loc_comparison.sort_values('diff', ascending=False)

print("=" * 60)
print("📊 按 Location 的准确率对比")
print("=" * 60)
print(f"{'Location':<15} {'Total':>8} {'Base Acc':>12} {'LoRA Acc':>12} {'Diff':>10}")
print("-" * 60)
for _, row in loc_comparison.iterrows():
    diff_str = f"{row['diff']:+.2%}"
    if row['diff'] > 0:
        diff_str = f"🟢 {diff_str}"
    elif row['diff'] < 0:
        diff_str = f"🔴 {diff_str}"
    print(f"{row['location']:<15} {row['total']:>8} {row['base_acc']:>12.2%} {row['lora_acc']:>12.2%} {diff_str:>15}")

## 8. 总结

In [ ]:
print("=" * 70)
print("📊 Common Base vs LoRA 分析总结")
print("=" * 70)

print(f"\n🎯 整体性能:")
print(f"   Base Accuracy: {base_metrics['accuracy']:.2%}")
print(f"   LoRA Accuracy: {lora_metrics['accuracy']:.2%}")
print(f"   差异: {diff_acc:+.2%}")

print(f"\n📈 样本级变化:")
print(f"   两模型都正确: {len(both_correct)} ({len(both_correct)/len(merged)*100:.1f}%)")
print(f"   两模型都错误: {len(both_wrong)} ({len(both_wrong)/len(merged)*100:.1f}%)")
print(f"   LoRA 改进 (Base错→LoRA对): {len(lora_only_correct)} ({len(lora_only_correct)/len(merged)*100:.1f}%)")
print(f"   LoRA 退步 (Base对→LoRA错): {len(base_only_correct)} ({len(base_only_correct)/len(merged)*100:.1f}%)")
print(f"   净改进: {len(lora_only_correct) - len(base_only_correct):+d} 样本")

print(f"\n📊 Category-Location Pairs:")
print(f"   改进的 pairs: {len(improved)}")
print(f"   退步的 pairs: {len(degraded)}")
print(f"   无变化的 pairs: {len(unchanged)}")

if diff_acc > 0:
    print(f"\n✅ 结论: LoRA 微调在 Common 图片上提升了 {diff_acc:.2%} 的准确率")
elif diff_acc < 0:
    print(f"\n⚠️ 结论: LoRA 微调在 Common 图片上降低了 {abs(diff_acc):.2%} 的准确率")
else:
    print(f"\n⚪ 结论: LoRA 微调在 Common 图片上准确率无变化")